# Employee Salary Prediction - Model Training

## Objective

The objective of this notebook is to build, train, and evaluate multiple machine learning regression models for predicting employee salaries.

### Models Used

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

### Evaluation Metrics

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np
import joblib

In [4]:
data = pd.read_csv("../data/employees_dataset_cleaned.csv")
df = pd.DataFrame(data)

### 2. Load the Cleaned Dataset

In [5]:
df

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069
...,...,...,...,...,...,...,...,...,...,...
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467


## 3. Separate Features and Target Variable

The target variable for this project is **Salary**, while all remaining columns are used as input features.

In [9]:
x = df.drop("salary", axis=1)

In [12]:
y = df["salary"]

### 4. Identify Numerical and Categorical Features

To apply appropriate preprocessing techniques, the dataset is divided into numerical and categorical features.

In [13]:
num_features = ["experience_years", "skills_count", "certifications"]

In [14]:
cat_features = ["job_title", "education_level", "industry", "company_size", "location", "remote_work"]

In [15]:
print(x.head())

            job_title  experience_years education_level  skills_count  \
0         AI Engineer                10        Bachelor             2   
1        Data Analyst                 5        Bachelor            17   
2  Frontend Developer                18             PhD             4   
3    Business Analyst                19             PhD            13   
4     Product Manager                15        Bachelor             7   

        industry company_size   location remote_work  certifications  
0     Healthcare       Medium      India      Hybrid               2  
1        Telecom        Small  Australia          No               0  
2          Media       Medium  Singapore          No               1  
3         Retail       Medium     Canada         Yes               0  
4  Manufacturing        Large     Sweden         Yes               0  


In [16]:
print(y.head())

0    109413
1     93764
2    148123
3    189123
4    165069
Name: salary, dtype: int64


In [17]:
print(x.dtypes)

job_title           object
experience_years     int64
education_level     object
skills_count         int64
industry            object
company_size        object
location            object
remote_work         object
certifications       int64
dtype: object


In [18]:
SimpleImputer(strategy="median")

SimpleImputer(strategy='median')

### 5. Split Dataset into Training and Testing Sets

The dataset is divided into training and testing subsets using an 80:20 ratio.

- Training Set: 80%
- Testing Set: 20%

In [19]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [20]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(200000, 9)
(50000, 9)
(200000,)
(50000,)


## 6. Data Preprocessing Pipeline

### Numerical Pipeline
Numerical features are:

- Imputed using the median strategy
- Standardized using StandardScaler

In [22]:
# Numerical Pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Categorical Pipeline
Categorical features are encoded using OneHotEncoder to convert text values into numerical representations.

In [23]:
# Categorical Pipeline
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### Combine Pipelines using ColumnTransformer

In [24]:
# Creating Full Pipeline
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

### 7. Transform Training and Testing Data

In [25]:
x_train_prepared = preprocessor.fit_transform(x_train)

In [26]:
print(x_train_prepared.shape)

(200000, 48)


In [27]:
x_test_prepared = preprocessor.transform(x_test)

In [28]:
print(x_test_prepared.shape)

(50000, 48)


## Train Model

### 8. Linear Regression
Linear Regression is used as the baseline model for salary prediction.

In [29]:
lin_reg = LinearRegression()

In [30]:
lin_reg.fit(x_train_prepared, y_train)

LinearRegression()

In [31]:
salary_predictions = lin_reg.predict(x_test_prepared)

In [32]:
print(salary_predictions)

[172850.84694068  89234.51854182  63791.55134444 ... 165227.44895727
 151849.72558748 111755.77171383]


### Model Evaluation
The model is evaluated using:

- MAE
- MSE
- RMSE
- R² Score

In [33]:
mae = mean_absolute_error(y_test, salary_predictions)
mse = mean_squared_error(y_test, salary_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, salary_predictions)

In [34]:
print("Mean Absolute Error :", mae)
print("Root Mean Square Error :", rmse)
print("R² :", r2)

Mean Absolute Error : 5436.096958649476
Root Mean Square Error : 7125.522920575375
R² : 0.9634690226760201


### 9. Decision Tree Regressor
Decision Trees learn hierarchical decision rules from the training data.

In [36]:
dec_reg = DecisionTreeRegressor()
dec_reg.fit(x_train_prepared, y_train)

DecisionTreeRegressor()

In [37]:
salary_predictions = dec_reg.predict(x_test_prepared)

In [38]:
print(salary_predictions)

[165943.  83825.  61482. ... 150685. 140999. 118571.]


#### Evaluate Dicision Tree Regressor Model

In [39]:
mae = mean_absolute_error(y_test, salary_predictions)
mse = mean_squared_error(y_test, salary_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, salary_predictions)

In [40]:
print("Mean Absolute Error :", mae)
print("Root Mean Square Error :", rmse)
print("R² :", r2)

Mean Absolute Error : 7279.01777
Root Mean Square Error : 9231.650679277514
R² : 0.9386822043627017


### 10. Random Forest Regressor
Random Forest combines multiple decision trees to improve prediction accuracy and reduce overfitting.

In [41]:
random_forest_reg = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)

In [42]:
random_forest_reg.fit(x_train_prepared, y_train)

RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)

In [43]:
salary_predictions = random_forest_reg.predict(x_test_prepared)

In [44]:
print(salary_predictions)

[167861.68  92881.16  70176.   ... 155343.02 148852.22 120405.26]


#### Evaluate Random Forest Regressor

In [45]:
mae = mean_absolute_error(y_test, salary_predictions)
mse = mean_squared_error(y_test, salary_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, salary_predictions)

In [46]:
print("Mean Absolute Error :", mae)
print("Root Mean Square Error :", rmse)
print("R² :", r2)

Mean Absolute Error : 5048.546023452857
Root Mean Square Error : 6364.544987946304
R² : 0.9708551026755426


## 11. Model Comparison

Performance of all trained models is compared using common regression metrics.

In [47]:
results = pd.DataFrame({
    "Model": ["Linear Regression","Decision Tree","Random Forest"],
    "MAE": [5436.09,7291.96,5048.55],
    "RMSE": [7125.52,9251.13,6364.54],
    "R2 Score": [0.96346,0.93842,0.97085]
})
results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,5436.09,7125.52,0.96346
1,Decision Tree,7291.96,9251.13,0.93842
2,Random Forest,5048.55,6364.54,0.97085


## 12. Best Performing Model

Based on the evaluation metrics, the **Random Forest Regressor** achieved the best overall performance among all the trained models.

### Performance Summary
| Metric | Random Forest |
|---------|--------------:|
| Mean Absolute Error (MAE) | **5048.55** |
| Root Mean Squared Error (RMSE) | **6364.54** |
| R² Score | **0.9709** |

### Conclusion

The Random Forest Regressor outperformed both Linear Regression and Decision Tree Regressor.

- ✅ Achieved the **highest R² Score (0.9709)**, indicating that it explains approximately **97% of the variance** in employee salaries.
- ✅ Produced the **lowest Mean Absolute Error (MAE)**, meaning its predictions are, on average, closest to the actual salaries.
- ✅ Recorded the **lowest Root Mean Squared Error (RMSE)**, demonstrating better overall prediction accuracy and fewer large prediction errors.

Therefore, the **Random Forest Regressor** is selected as the **best baseline model** for this project and will be used in the next phase for **hyperparameter tuning using RandomizedSearchCV** to further improve its predictive performance.

In [27]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions={
        "n_estimators": [50,100,150,200],
        "max_depth": [10,20,30,None],
        "min_samples_split": [2,5,10],
        "min_samples_leaf": [1,2,4]
    },
    n_iter=10,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train_prepared, y_train)

RandomizedSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, 30, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 150, 200]},
                   random_state=42, scoring='r2')

In [29]:
print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 30}


In [30]:
print("Best Cross Validation Score:")
print(random_search.best_score_)

Best Cross Validation Score:
0.9676678268521144


## 13. Evaluation of Tuned Random Forest

The best-performing hyperparameter combination obtained through RandomizedSearchCV was used to create the optimized Random Forest model.

The tuned model was then evaluated on the unseen test dataset using the same evaluation metrics (MAE, RMSE, and R² Score). This allows a fair comparison between the default Random Forest model and the optimized version to determine which one generalizes better.

In [31]:
best_model = random_search.best_estimator_

In [32]:
print(best_model)

RandomForestRegressor(max_depth=30, min_samples_split=10, n_estimators=150,
                      random_state=42)


In [34]:
predictions = best_model.predict(x_test_prepared)

In [35]:
mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

MAE : 5097.757087881144
RMSE : 6437.156278809632
R² : 0.9701862974549542


## 14. Final Model Selection

After evaluating both models on the test dataset, the original Random Forest Regressor achieved slightly better performance than the tuned model.

Although RandomizedSearchCV identified an optimized set of hyperparameters through cross-validation, these settings did not improve the model's performance on unseen data.

Therefore, the original Random Forest Regressor was selected as the final production model because it achieved:

- Lower Mean Absolute Error (MAE)
- Lower Root Mean Squared Error (RMSE)
- Higher R² Score

This demonstrates that hyperparameter tuning does not always guarantee better real-world performance. The final model should always be selected based on evaluation using the held-out test dataset.

## 15. Actual vs Predicted Salary Comparison

To further evaluate the performance of the selected model, a comparison is made between the actual salary values and the salaries predicted by the Random Forest Regressor.

The following table displays:

- **Actual Salary** – The true salary from the test dataset.
- **Predicted Salary** – The salary estimated by the trained model.
- **Difference** – The prediction error (Actual − Predicted).
- **Percentage Error** – The relative prediction error expressed as a percentage.

This comparison provides a practical understanding of how closely the model's predictions match the actual salary values.

In [48]:
comparison = pd.DataFrame({"Actual Salary": y_test.values, "Predicted Salary": salary_predictions})

In [49]:
comparison["Difference"] = (comparison["Actual Salary"] -comparison["Predicted Salary"])
comparison["Percentage Error"] = (abs(comparison["Difference"])/comparison["Actual Salary"]) * 100
comparison.head(15)

,Actual Salary,Predicted Salary,Difference,Percentage Error
0,164009,167861.68,-3852.68,2.349066
1,79594,92881.16,-13287.16,16.693670
2,74090,70176.00,3914.00,5.282764
3,177193,162563.08,14629.92,8.256489
4,120012,115070.18,4941.82,4.117772
5,163369,163882.26,-513.26,0.314172
6,111889,110474.78,1414.22,1.263949
7,75418,73300.76,2117.24,2.807340
8,103067,96603.02,6463.98,6.271629
9,190692,196073.36,-5381.36,2.822017


In [50]:
comparison.to_csv("../data/salary_predictions.csv",index=False)

# 16. Saving the Final Model

## Objective

After comparing multiple regression models and evaluating the tuned Random Forest model, the original Random Forest Regressor was selected as the final production model.

To make the model reusable without retraining, the trained model and preprocessing pipeline are saved using **Joblib**.

The following files are stored inside the `models/` directory:

- **random_forest_salary_model.pkl** – Trained Random Forest model.
- **preprocessor.pkl** – Data preprocessing pipeline used during training.

These files will be loaded later in the Streamlit application to perform salary predictions on new employee data.

In [51]:
joblib.dump(random_forest_reg, "../models/random_forest_salary_model.pkl")

['../models/random_forest_salary_model.pkl']

In [52]:
joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']